# Cash Flow Runway Advisor
**Course:** Generative AI &nbsp;|&nbsp; **Track:** Claude Agent SDK (Track B) &nbsp;|&nbsp; **Group 4:** Xing Wang, Yi Lu, Aastik Rajan

Production-grade agentic system that helps a founder/CFO answer cash-flow questions like:
- *Can I afford to hire a $140k engineer in Q2?*
- *If MegaCorp pays 45 days late, do I still have safe runway?*
- *Should I cut marketing 30% to extend runway?*

Architecture: a **Planner** (Claude Sonnet 4.6) decomposes the request into a JSON plan; an **Executor** (Claude Haiku 4.5) drives a manual `tool_use` loop over three deterministic tools; a separate **Judge** (Sonnet 4.6) scores each run against a 6-axis rubric with required citations. Hard guardrails on iterations, tokens, wall-clock, and tool timeouts. Full telemetry is captured per task.

## 1. Setup
Installs dependencies and pulls the API key from Colab secrets (`ANTHROPIC_API_KEY`). Run this cell first.

In [ ]:
%%capture
%pip install --quiet anthropic==0.77.1 pandas


In [ ]:
import os

# Pull API key: prefer Colab userdata, fall back to env var.
try:
    from google.colab import userdata  # type: ignore
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
    print('API key loaded from Colab userdata.')
except Exception:
    if not os.environ.get('ANTHROPIC_API_KEY'):
        raise RuntimeError('Set ANTHROPIC_API_KEY in Colab secrets or env.')
    print('API key loaded from environment.')

from anthropic import Anthropic
client = Anthropic()
print('Anthropic client ready.')


API key loaded from Colab userdata.
Anthropic client ready.


## 2. Synthetic Accounting Data (OrbitSaaS, 18 months)
Why synthetic: privacy, plus we can plant analytically useful patterns the agent can *discover*:
- **MegaCorp Industries** ≈ 48% of revenue, paying ~38 days late on average (concentration risk).
- Summer revenue dip (Jun-Aug), Q4 spike on both sales and marketing spend.
- Two one-time expense shocks (legal settlement m8, office expansion m12).
- Linear headcount growth lifting recurring salary expense.

The seed (`SEED=42`) is fixed so evaluation runs are reproducible — variance reflects agent behavior, not data drift.

In [ ]:
"""Synthetic accounting data generator for OrbitSaaS, a fictional B2B SaaS company.

Generates 18 months of:
  - invoices (with intentional late-payment patterns)
  - expenses (salaries, software, marketing, legal, office, with seasonality)
  - cash_snapshots (month-end balances)
  - customers (tiers, payment behavior)

Patterns intentionally planted so the agent has something to discover:
  - MegaCorp: ~40% of revenue, pays 30-45 days late on average
  - Summer revenue dip (Jun-Aug), Q4 spike
  - One-time legal expense shock in month 8
  - Gradual headcount growth (rising salary expense)
"""

from __future__ import annotations

import random
import sqlite3
from dataclasses import dataclass
from datetime import date, timedelta
from pathlib import Path

import pandas as pd

SEED = 42
START_DATE = date(2024, 11, 1)  # 18 months ending 2026-04-30
MONTHS = 18

CUSTOMERS = [
    # (name, tier, base_monthly_arr, avg_late_days, lateness_jitter)
    ("MegaCorp Industries", "enterprise", 42_000, 38, 7),  # the planted whale
    ("Northwind Labs",      "mid_market", 12_000, 8,  4),
    ("Acme Robotics",       "mid_market", 9_500,  5,  3),
    ("Bluefin Analytics",   "mid_market", 8_000,  10, 5),
    ("Pioneer Health",      "smb",        4_500,  3,  2),
    ("Cobalt Finance",      "smb",        3_800,  2,  2),
    ("Quanta Logistics",    "smb",        3_200,  6,  3),
    ("Helio Studios",       "smb",        2_400,  4,  3),
    ("Drift Mobile",        "smb",        2_100,  1,  2),
    ("Lumen Retail",        "smb",        1_800,  2,  2),
]

EXPENSE_CATEGORIES = {
    "salaries":  {"base": 62_000, "growth": 1_800, "variance": 1_500, "recurring": True},
    "software":  {"base": 6_500,  "growth": 120,   "variance": 800,   "recurring": True},
    "marketing": {"base": 8_000,  "growth": 200,   "variance": 2_400, "recurring": True},
    "office":    {"base": 4_200,  "growth": 50,    "variance": 600,   "recurring": True},
    "legal":     {"base": 1_200,  "growth": 0,     "variance": 400,   "recurring": False},
    "infra":     {"base": 5_800,  "growth": 250,   "variance": 700,   "recurring": True},
}

# Plant a one-time expense shock in month 8 (legal settlement)
ONE_TIME_SHOCKS = [
    (8, "legal",   45_000, "Trademark dispute settlement"),
    (12, "office", 18_000, "Office expansion deposit"),
]

OPENING_CASH = 720_000  # starting cash balance

def _seasonal_multiplier(month_index: int) -> float:
    """Revenue seasonality: dip in summer, Q4 spike."""
    cal_month = (START_DATE.month - 1 + month_index) % 12 + 1
    if cal_month in (6, 7, 8):    # summer dip
        return 0.88
    if cal_month in (10, 11, 12): # Q4 spike
        return 1.12
    return 1.0

def _expense_seasonal(month_index: int, category: str) -> float:
    cal_month = (START_DATE.month - 1 + month_index) % 12 + 1
    if category == "marketing" and cal_month in (10, 11, 12):
        return 1.35  # Q4 marketing push
    if category == "marketing" and cal_month in (6, 7, 8):
        return 0.75
    return 1.0

def _month_end(month_index: int) -> date:
    """End-of-month date for the i-th month after START_DATE."""
    year = START_DATE.year + (START_DATE.month - 1 + month_index + 1) // 12
    month = (START_DATE.month - 1 + month_index + 1) % 12 + 1
    if month == 1:
        return date(year, 1, 1) - timedelta(days=1)
    return date(year, month, 1) - timedelta(days=1)

def _month_start(month_index: int) -> date:
    year = START_DATE.year + (START_DATE.month - 1 + month_index) // 12
    month = (START_DATE.month - 1 + month_index) % 12 + 1
    return date(year, month, 1)

@dataclass
class FinancialDataset:
    invoices: pd.DataFrame
    expenses: pd.DataFrame
    cash_snapshots: pd.DataFrame
    customers: pd.DataFrame

    def to_sqlite(self, path: str | Path = ":memory:") -> sqlite3.Connection:
        conn = sqlite3.connect(str(path),check_same_thread=False)
        self.invoices.to_sql("invoices", conn, if_exists="replace", index=False)
        self.expenses.to_sql("expenses", conn, if_exists="replace", index=False)
        self.cash_snapshots.to_sql("cash_snapshots", conn, if_exists="replace", index=False)
        self.customers.to_sql("customers", conn, if_exists="replace", index=False)
        conn.commit()
        return conn

def generate(seed: int = SEED) -> FinancialDataset:
    rng = random.Random(seed)

    # ----- customers
    customers_rows = []
    for cid, (name, tier, arr, late, jitter) in enumerate(CUSTOMERS, start=1):
        customers_rows.append({
            "customer_id": cid,
            "name": name,
            "tier": tier,
            "monthly_arr": arr,
            "avg_payment_days_late": late,
            "lateness_jitter": jitter,
        })
    customers_df = pd.DataFrame(customers_rows)

    # ----- invoices: one per customer per month
    invoices_rows = []
    invoice_id = 1000
    for m in range(MONTHS):
        seasonal = _seasonal_multiplier(m)
        for cid, (name, tier, arr, late, jitter) in enumerate(CUSTOMERS, start=1):
            issue = _month_start(m)
            due = issue + timedelta(days=30)
            amount_noise = rng.uniform(0.92, 1.08)
            amount = round(arr * seasonal * amount_noise, 2)

            # paid_date and status; current month invoices may still be unpaid
            late_days = max(0, int(rng.gauss(late, max(jitter, 1))))
            target_pay = due + timedelta(days=late_days)
            today = _month_end(MONTHS - 1)  # "as of" the last data point

            if target_pay <= today:
                paid_date = target_pay
                status = "paid"
            else:
                paid_date = None
                # if past due but unpaid -> outstanding/late, else open
                status = "late" if due < today else "open"

            invoices_rows.append({
                "invoice_id": invoice_id,
                "customer_id": cid,
                "customer_name": name,
                "amount": amount,
                "issue_date": issue.isoformat(),
                "due_date": due.isoformat(),
                "paid_date": paid_date.isoformat() if paid_date else None,
                "status": status,
            })
            invoice_id += 1
    invoices_df = pd.DataFrame(invoices_rows)

    # ----- expenses
    expenses_rows = []
    expense_id = 5000
    for m in range(MONTHS):
        for cat, spec in EXPENSE_CATEGORIES.items():
            seasonal = _expense_seasonal(m, cat)
            base = spec["base"] + spec["growth"] * m
            noise = rng.gauss(0, spec["variance"])
            amount = max(0, round((base + noise) * seasonal, 2))
            txn_date = _month_start(m) + timedelta(days=rng.randint(0, 27))
            expenses_rows.append({
                "expense_id": expense_id,
                "date": txn_date.isoformat(),
                "category": cat,
                "amount": amount,
                "description": f"{cat.title()} - month {m+1}",
                "is_recurring": int(spec["recurring"]),
            })
            expense_id += 1
        # one-time shocks
        for shock_month, shock_cat, shock_amt, shock_desc in ONE_TIME_SHOCKS:
            if shock_month == m:
                expenses_rows.append({
                    "expense_id": expense_id,
                    "date": (_month_start(m) + timedelta(days=15)).isoformat(),
                    "category": shock_cat,
                    "amount": shock_amt,
                    "description": shock_desc,
                    "is_recurring": 0,
                })
                expense_id += 1
    expenses_df = pd.DataFrame(expenses_rows)

    # ----- cash snapshots: month-end, derived from opening + collections - expenses
    cash_rows = []
    cash = OPENING_CASH
    invoices_df["_paid_dt"] = pd.to_datetime(invoices_df["paid_date"])
    expenses_df["_dt"] = pd.to_datetime(expenses_df["date"])
    for m in range(MONTHS):
        ms, me = _month_start(m), _month_end(m)
        collected = invoices_df[
            invoices_df["_paid_dt"].between(pd.Timestamp(ms), pd.Timestamp(me))
        ]["amount"].sum()
        spent = expenses_df[
            expenses_df["_dt"].between(pd.Timestamp(ms), pd.Timestamp(me))
        ]["amount"].sum()
        cash = cash + float(collected) - float(spent)
        cash_rows.append({
            "snapshot_date": me.isoformat(),
            "cash_balance": round(cash, 2),
            "collections_in_month": round(float(collected), 2),
            "expenses_in_month": round(float(spent), 2),
        })
    cash_df = pd.DataFrame(cash_rows)

    invoices_df = invoices_df.drop(columns=["_paid_dt"])
    expenses_df = expenses_df.drop(columns=["_dt"])

    return FinancialDataset(
        invoices=invoices_df,
        expenses=expenses_df,
        cash_snapshots=cash_df,
        customers=customers_df,
    )


In [ ]:
_ds = generate()
_conn = _ds.to_sqlite()
print(f'Invoices:        {len(_ds.invoices):>4}  (status: {_ds.invoices.status.value_counts().to_dict()})')
print(f'Expenses:        {len(_ds.expenses):>4}')
print(f'Cash snapshots:  {len(_ds.cash_snapshots):>4}')
print(f'Customers:       {len(_ds.customers):>4}')
print()
print('Last 3 month-end cash snapshots:')
print(_ds.cash_snapshots.tail(3).to_string(index=False))
print()
_total = _ds.invoices['amount'].sum()
_mega = _ds.invoices[_ds.invoices.customer_name == 'MegaCorp Industries']['amount'].sum()
print(f"MegaCorp share of total revenue: {_mega/_total*100:.1f}%  ({_mega:,.0f} of {_total:,.0f})")


Invoices:         180  (status: {'paid': 169, 'open': 10, 'late': 1})
Expenses:         110
Cash snapshots:    18
Customers:         10

Last 3 month-end cash snapshots:
snapshot_date  cash_balance  collections_in_month  expenses_in_month
   2026-02-28     292672.04              96499.99          126825.83
   2026-03-31     308045.98             141161.64          125787.70
   2026-04-30     220778.18              42670.63          129938.43

MegaCorp share of total revenue: 47.9%  (789,210 of 1,649,108)


## 3. Tools (Three Deterministic Functions)
Per ADR *AI vs Rule-Based*: numeric calculations live here, not in the LLM.

1. **`query_financials(sql)`** — read-only SQL over the four tables. Defense-in-depth rejects anything that isn't a single `SELECT` (or `WITH … SELECT`).
2. **`calculate_runway(...)`** — runway in months from a rolling 3-month burn window with low/high sensitivity bounds derived from burn σ. Accepts `scenario_deltas` for inline what-ifs.
3. **`simulate_scenario(action, params)`** — named what-ifs (`hire`, `delay_hire`, `cut_spend`, `ar_delay`, `price_increase`, `raise_capital`). Returns a structured *baseline vs scenario vs delta vs verdict* comparison so the agent interprets the trade-off rather than computing it.

Every tool returns `{status: ok|error, ...}` so the agent can recover from bad input instead of crashing the loop.

In [ ]:
"""Three deterministic tools the agent will call.

Design rules:
  1. Every tool returns structured JSON-serializable output (never just a string).
  2. Every tool validates inputs and returns an explicit error envelope on bad input
     so the agent can recover instead of crashing the loop.
  3. Numeric calculations live HERE, not in the LLM (per ADR: AI vs Rule-Based).
"""

from __future__ import annotations

import sqlite3
from dataclasses import dataclass
from datetime import date, timedelta
from typing import Any

import pandas as pd

# ----------- shared helpers --------------------------------------------------

def _ok(**payload) -> dict:
    return {"status": "ok", **payload}

def _err(message: str, **extra) -> dict:
    return {"status": "error", "error": message, **extra}

def _safe_select(sql: str) -> bool:
    """Reject anything that isn't a single SELECT statement.

    Defense-in-depth: even though the agent is supposed to behave, we don't want
    a hallucinated DROP/UPDATE to corrupt the in-memory database.
    """
    s = sql.strip().rstrip(";").lower()
    if ";" in s:
        return False
    if not s.startswith(("select", "with ")):
        return False
    forbidden = ("insert", "update", "delete", "drop", "alter", "create",
                 "attach", "pragma", "replace ", "vacuum")
    return not any(tok in s for tok in forbidden)

# ----------- Tool 1: query_financials ---------------------------------------

def query_financials(conn: sqlite3.Connection, sql: str, max_rows: int = 50) -> dict:
    """Run a read-only SQL query against the in-memory accounting DB.

    Tables available:
      invoices(invoice_id, customer_id, customer_name, amount,
               issue_date, due_date, paid_date, status)
      expenses(expense_id, date, category, amount, description, is_recurring)
      cash_snapshots(snapshot_date, cash_balance,
                     collections_in_month, expenses_in_month)
      customers(customer_id, name, tier, monthly_arr,
                avg_payment_days_late, lateness_jitter)
    """
    if not isinstance(sql, str) or not sql.strip():
        return _err("empty SQL")
    if not _safe_select(sql):
        return _err("only single SELECT statements are permitted")
    try:
        df = pd.read_sql_query(sql, conn)
    except Exception as e:
        return _err(f"SQL execution failed: {e}", sql=sql)

    truncated = len(df) > max_rows
    if truncated:
        df = df.head(max_rows)
    rows = df.to_dict(orient="records")
    # Round floats so the LLM doesn't choke on noisy decimals
    for r in rows:
        for k, v in list(r.items()):
            if isinstance(v, float):
                r[k] = round(v, 2)
    return _ok(rows=rows, row_count=len(rows), truncated=truncated, columns=list(df.columns))

# ----------- Tool 2: calculate_runway ---------------------------------------

@dataclass
class RunwayResult:
    cash_balance: float
    monthly_burn: float
    monthly_collections: float
    monthly_net_burn: float
    runway_months: float
    runway_low: float       # pessimistic (burn + 1 std)
    runway_high: float      # optimistic (burn - 1 std)
    burn_stdev: float
    burn_window_months: int
    safety_threshold_months: float
    below_safety_threshold: bool

    def to_dict(self) -> dict:
        return self.__dict__

def calculate_runway(
    conn: sqlite3.Connection,
    cash_balance: float | None = None,
    monthly_burn: float | None = None,
    scenario_deltas: dict | None = None,
    burn_window_months: int = 3,
    safety_threshold_months: float = 6.0,
) -> dict:
    """Compute runway off a rolling burn window, with sensitivity bounds.

    Conventions:
      monthly_burn       = average gross expenses over the trailing window
      monthly_collections= average collections over the same window
      monthly_net_burn   = monthly_burn - monthly_collections
      runway_months      = cash / monthly_net_burn   (clipped at 0/inf)

    scenario_deltas may include any of:
      cash_delta              -> add to cash_balance
      monthly_expense_delta   -> add to monthly_burn (e.g. +11_667 for $140k hire)
      monthly_collection_delta-> add to monthly_collections
      ar_delay_days           -> defer N days of collections (haircut)
    """
    try:
        snap = pd.read_sql_query(
            "SELECT * FROM cash_snapshots ORDER BY snapshot_date DESC", conn
        )
    except Exception as e:
        return _err(f"could not read cash_snapshots: {e}")

    if snap.empty:
        return _err("no cash snapshots available")

    if cash_balance is None:
        cash_balance = float(snap.iloc[0]["cash_balance"])

    window = snap.head(max(1, burn_window_months))
    window_collections = window["collections_in_month"].astype(float)
    window_expenses = window["expenses_in_month"].astype(float)

    avg_collections = float(window_collections.mean())
    if monthly_burn is None:
        monthly_burn = float(window_expenses.mean())
    burn_stdev = float(window_expenses.std(ddof=0)) if len(window_expenses) > 1 else 0.0

    deltas = scenario_deltas or {}
    if not isinstance(deltas, dict):
        return _err("scenario_deltas must be an object")

    cash_balance += float(deltas.get("cash_delta", 0) or 0)
    monthly_burn += float(deltas.get("monthly_expense_delta", 0) or 0)
    avg_collections += float(deltas.get("monthly_collection_delta", 0) or 0)

    ar_delay = float(deltas.get("ar_delay_days", 0) or 0)
    if ar_delay > 0:
        # AR delay is a timing issue, not a revenue loss: customers eventually
        # pay, but the cash arrives ar_delay days later. Model as a one-time
        # hit to opening cash equal to ar_delay/30 months of collections.
        # Average go-forward collections are unchanged.
        deferred_cash = avg_collections * (ar_delay / 30.0)
        cash_balance -= deferred_cash

    net_burn = monthly_burn - avg_collections

    def _runway(net):
        if net <= 0:
            return float("inf")
        return cash_balance / net

    runway = _runway(net_burn)
    runway_low = _runway(net_burn + burn_stdev)
    runway_high = _runway(max(0.01, net_burn - burn_stdev))

    res = RunwayResult(
        cash_balance=round(cash_balance, 2),
        monthly_burn=round(monthly_burn, 2),
        monthly_collections=round(avg_collections, 2),
        monthly_net_burn=round(net_burn, 2),
        runway_months=round(runway, 2) if runway != float("inf") else None,
        runway_low=round(runway_low, 2) if runway_low != float("inf") else None,
        runway_high=round(runway_high, 2) if runway_high != float("inf") else None,
        burn_stdev=round(burn_stdev, 2),
        burn_window_months=burn_window_months,
        safety_threshold_months=safety_threshold_months,
        below_safety_threshold=(runway < safety_threshold_months),
    )
    return _ok(**res.to_dict())

# ----------- Tool 3: simulate_scenario --------------------------------------

KNOWN_ACTIONS = {
    "hire":            "Add headcount; params: salary_annual, start_month_offset (default 0)",
    "delay_hire":      "Postpone a hire; params: salary_annual, delay_months",
    "cut_spend":       "Cut a recurring category by %; params: category, pct",
    "ar_delay":        "Major customer pays late; params: days, customer_share (0-1)",
    "price_increase":  "Raise prices; params: pct (applies to monthly collections)",
    "raise_capital":   "One-time injection; params: amount",
}

def simulate_scenario(
    conn: sqlite3.Connection,
    action: str,
    params: dict | None = None,
    timeframe_months: int = 12,
) -> dict:
    """Compare a baseline runway vs an action-modified runway.

    Returns a structured comparison so the agent can interpret the trade-off
    instead of guessing at numbers itself.
    """
    if action not in KNOWN_ACTIONS:
        return _err(
            f"unknown action '{action}'",
            known_actions=KNOWN_ACTIONS,
        )
    params = params or {}
    deltas: dict[str, float] = {}
    description = ""
    growth_risk: dict = {}

    if action == "hire":
        salary = float(params.get("salary_annual", 0))
        if salary <= 0:
            return _err("hire requires salary_annual > 0")
        monthly = salary / 12
        deltas["monthly_expense_delta"] = monthly
        description = f"Hire at ${salary:,.0f}/yr (+${monthly:,.0f}/month burn)"

    elif action == "delay_hire":
        salary = float(params.get("salary_annual", 0))
        delay = int(params.get("delay_months", 3))
        if salary <= 0 or delay <= 0:
            return _err("delay_hire requires salary_annual > 0 and delay_months > 0")
        # Average monthly impact over the timeframe is reduced
        months_active = max(0, timeframe_months - delay)
        avg_monthly = (salary / 12) * (months_active / timeframe_months)
        deltas["monthly_expense_delta"] = avg_monthly
        description = (
            f"Delay hire {delay} months; avg incremental burn over {timeframe_months}mo = "
            f"${avg_monthly:,.0f}/month"
        )

    elif action == "cut_spend":
        # Need to look up current avg category spend to compute the cut value.
        category = str(params.get("category", "")).lower()
        pct = float(params.get("pct", 0))
        if not category or pct <= 0:
            return _err("cut_spend requires category and pct > 0")
        df = pd.read_sql_query(
            "SELECT category, AVG(amount) AS avg_amt "
            "FROM expenses WHERE is_recurring = 1 GROUP BY category",
            conn,
        )
        match = df[df["category"].str.lower() == category]
        if match.empty:
            return _err(
                f"category '{category}' not found in recurring expenses",
                available=df["category"].tolist(),
            )
        cur_monthly = float(match.iloc[0]["avg_amt"])
        savings = cur_monthly * (pct / 100.0)
        deltas["monthly_expense_delta"] = -savings
        description = (
            f"Cut {category} by {pct:.0f}% (~${savings:,.0f}/month savings, "
            f"baseline ${cur_monthly:,.0f}/month)"
        )
        if category == "marketing":
            risk_pct = round((savings / 1_000) * 0.5, 1)
            growth_risk = {
                "growth_risk_pct_per_month": risk_pct,
                "pipeline_recovery_months": 3,
                "note": (
                    f"Cutting ${savings:,.0f}/mo from marketing is estimated to"
                    f" reduce MoM revenue growth by ~{risk_pct} pp for ~3 months"
                    f" (rule: $1k/mo cut ≈ 0.5 pp growth drag; ROI lag ≈ 3 months)."
                ),
            }

    elif action == "ar_delay":
        days = float(params.get("days", 30))
        share = float(params.get("customer_share", 0.4))  # MegaCorp is ~40%
        if days <= 0 or not (0 < share <= 1):
            return _err("ar_delay requires days > 0 and 0 < customer_share <= 1")
        deltas["ar_delay_days"] = days * share
        description = f"AR delay: {days:.0f} extra days on {share*100:.0f}% of revenue"

    elif action == "price_increase":
        pct = float(params.get("pct", 0))
        if pct <= 0:
            return _err("price_increase requires pct > 0")
        # Apply to current avg collections - need baseline first
        baseline = calculate_runway(conn)
        if baseline.get("status") != "ok":
            return baseline
        uplift = baseline["monthly_collections"] * (pct / 100.0)
        deltas["monthly_collection_delta"] = uplift
        description = f"Price increase {pct:.0f}% (+${uplift:,.0f}/month collections)"

    elif action == "raise_capital":
        amount = float(params.get("amount", 0))
        if amount <= 0:
            return _err("raise_capital requires amount > 0")
        deltas["cash_delta"] = amount
        description = f"Raise ${amount:,.0f} (one-time cash injection)"

    baseline = calculate_runway(conn)
    if baseline.get("status") != "ok":
        return baseline
    scenario = calculate_runway(conn, scenario_deltas=deltas)
    if scenario.get("status") != "ok":
        return scenario

    def _diff(key):
        b, s = baseline.get(key), scenario.get(key)
        if b is None or s is None:
            return None
        return round(s - b, 2)

    comparison = {
        "action": action,
        "description": description,
        "params": params,
        "deltas_applied": deltas,
        "baseline": {
            "runway_months": baseline["runway_months"],
            "monthly_net_burn": baseline["monthly_net_burn"],
            "cash_balance": baseline["cash_balance"],
        },
        "scenario": {
            "runway_months": scenario["runway_months"],
            "monthly_net_burn": scenario["monthly_net_burn"],
            "cash_balance": scenario["cash_balance"],
            "below_safety_threshold": scenario["below_safety_threshold"],
        },
        "delta": {
            "runway_months": _diff("runway_months"),
            "monthly_net_burn": _diff("monthly_net_burn"),
        },
        "verdict": _verdict(baseline, scenario),
        "growth_risk": growth_risk,
    }
    return _ok(**comparison)

def _verdict(baseline: dict, scenario: dict) -> str:
    b = baseline.get("runway_months")
    s = scenario.get("runway_months")
    safety = baseline.get("safety_threshold_months", 6)
    if b is None or s is None:
        return "indeterminate"
    if scenario.get("below_safety_threshold") and not baseline.get("below_safety_threshold"):
        return "warning_breaches_safety_threshold"
    if s >= b:
        return "improves_or_neutral"
    if s >= safety:
        return "acceptable_but_shorter_runway"
    return "warning_below_safety_threshold"

# ----------- Anthropic tool schemas (single source of truth) -----------------

TOOL_SCHEMAS = [
    {
        "name": "query_financials",
        "description": (
            "Run a read-only SQL SELECT against the in-memory accounting database. "
            "Tables: invoices, expenses, cash_snapshots, customers. "
            "Use this when you need raw numbers, lists, or aggregates. "
            "Only one SELECT (or WITH...SELECT) statement at a time. "
            "Always quote table/column names exactly as listed."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "sql": {
                    "type": "string",
                    "description": "A single SQLite SELECT statement.",
                },
            },
            "required": ["sql"],
        },
    },
    {
        "name": "calculate_runway",
        "description": (
            "Deterministically compute cash runway in months from the latest "
            "cash snapshot and a rolling 3-month burn window. Optionally apply "
            "scenario_deltas to model what-ifs WITHOUT calling simulate_scenario. "
            "Returns runway_months plus low/high sensitivity bounds. "
            "Prefer simulate_scenario for named actions like hire/cut_spend; use this "
            "tool when you want raw numeric control."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "cash_balance": {"type": "number", "description": "Override cash; default = latest snapshot."},
                "monthly_burn": {"type": "number", "description": "Override burn; default = trailing-window average."},
                "scenario_deltas": {
                    "type": "object",
                    "description": (
                        "Optional adjustments. Keys: cash_delta, "
                        "monthly_expense_delta, monthly_collection_delta, ar_delay_days."
                    ),
                },
                "burn_window_months": {"type": "integer", "description": "Trailing months to average. Default 3."},
                "safety_threshold_months": {"type": "number", "description": "Months considered 'safe'. Default 6."},
            },
        },
    },
    {
        "name": "simulate_scenario",
        "description": (
            "Compare the baseline runway against a named action. Returns a "
            "structured comparison (baseline vs scenario vs delta vs verdict). "
            "Use this for hire/delay_hire/cut_spend/ar_delay/price_increase/raise_capital."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "action": {
                    "type": "string",
                    "enum": list(KNOWN_ACTIONS.keys()),
                    "description": "Named scenario type.",
                },
                "params": {
                    "type": "object",
                    "description": (
                        "Action-specific parameters. "
                        "hire/delay_hire: salary_annual, delay_months. "
                        "cut_spend: category, pct. "
                        "ar_delay: days, customer_share. "
                        "price_increase: pct. "
                        "raise_capital: amount."
                    ),
                },
                "timeframe_months": {"type": "integer", "description": "Default 12."},
            },
            "required": ["action"],
        },
    },
]

def dispatch(conn: sqlite3.Connection, name: str, tool_input: dict) -> dict:
    """Single entry point used by the executor agent's tool_use loop."""
    try:
        if name == "query_financials":
            return query_financials(conn, **tool_input)
        if name == "calculate_runway":
            return calculate_runway(conn, **tool_input)
        if name == "simulate_scenario":
            return simulate_scenario(conn, **tool_input)
        return _err(f"unknown tool '{name}'")
    except TypeError as e:
        return _err(f"bad arguments for {name}: {e}", received=tool_input)
    except Exception as e:
        return _err(f"{name} crashed: {e}")


In [ ]:
# Quick deterministic sanity check before we burn API tokens.
import json as _json
print('Baseline runway:')
print(_json.dumps(calculate_runway(_conn), indent=2))
print()
print('Hire $140k engineer:')
print(_json.dumps(simulate_scenario(_conn, 'hire', {'salary_annual': 140000})['delta'], indent=2))
print()
print('Cut marketing 30%:')
print(_json.dumps(simulate_scenario(_conn, 'cut_spend', {'category': 'marketing', 'pct': 30})['delta'], indent=2))


Baseline runway:
{
  "status": "ok",
  "cash_balance": 220778.18,
  "monthly_burn": 127517.32,
  "monthly_collections": 93444.09,
  "monthly_net_burn": 34073.23,
  "runway_months": 6.48,
  "runway_low": 6.16,
  "runway_high": 6.83,
  "burn_stdev": 1763.66,
  "burn_window_months": 3,
  "safety_threshold_months": 6.0,
  "below_safety_threshold": false
}

Hire $140k engineer:
{
  "runway_months": -1.65,
  "monthly_net_burn": 11666.67
}

Cut marketing 30%:
{
  "runway_months": 0.67,
  "monthly_net_burn": -3187.12
}


## 4. Multi-Agent Orchestrator (Planner ➜ Executor)
Track B requirement: manual `tool_use` loop, multi-agent design.

**Planner (Sonnet 4.6)** — reads the user question and produces a JSON plan with one of three actions:
- `execute` (with ordered tool steps),
- `clarify` (with up to 3 clarifying questions),
- `out_of_scope` (when the request is outside the data window).

**Executor (Haiku 4.5)** — receives the question + plan and drives the manual `tool_use` loop, producing a final `<answer>…</answer>` block with cited numbers, explicit risks, and a recommendation. Forbidden from inventing numbers (per system prompt).

**Guardrails (hard kill switches)** — `max_iterations=7`, `max_total_tokens=15_000`, `tool_timeout_s=10.0`, `max_wall_seconds=60.0`. Each is enforced before the next loop iteration; trips are logged as `kind='guardrail'` steps in the trace.

**Telemetry (`Trace`)** — every planner thought, executor message, tool call, tool result, error, and guardrail breach is appended with a monotonic timestamp. The Judge consumes this trace verbatim.

**FinOps (`CostTracker`)** — input/output tokens × per-model price → USD per task. Inference vs tool wall time tracked separately so we can identify the bottleneck.

In [ ]:
"""Multi-agent orchestrator: Planner (Sonnet) -> Executor (Haiku) -> Answer.

Track B / Claude Agent SDK style: we manually drive the tool_use loop instead
of using a framework. The loop is deterministic and bounded.

Guardrails (hard kill switches):
  - max_iterations: caps the executor's tool_use cycles.
  - max_total_tokens: caps cumulative input+output tokens per task.
  - tool_timeout_s: per-tool wall clock cap.

Telemetry:
  - Every step (planner thought, tool call, tool result, model message, error)
    is appended to a Trace with monotonic timestamps. Persisted as JSON for
    the Judge agent and the FinOps report.

FinOps:
  - CostTracker estimates $ from input/output token counts using a per-model
    pricing table. Numbers are estimates; the user can swap in real list
    prices in PRICES.
"""

from __future__ import annotations

import json
import sqlite3
import threading
import time
import uuid
from dataclasses import dataclass, field
from typing import Any, Callable

from anthropic import Anthropic

# ----------------------------------------------------------------------------
# Pricing (USD per 1M tokens) -- swap with current list prices as needed.
# ----------------------------------------------------------------------------
PRICES = {
    "claude-opus-4-7":            {"input": 15.0, "output": 75.0},
    "claude-sonnet-4-6":          {"input": 3.0,  "output": 15.0},
    "claude-haiku-4-5-20251001":  {"input": 1.0,  "output": 5.0},
    # Fallback for anything else:
    "_default":                   {"input": 3.0,  "output": 15.0},
}

PLANNER_MODEL  = "claude-sonnet-4-6"
EXECUTOR_MODEL = "claude-haiku-4-5-20251001"
JUDGE_MODEL    = "claude-sonnet-4-6"

# ----------------------------------------------------------------------------
# Telemetry
# ----------------------------------------------------------------------------

@dataclass
class TraceStep:
    ts: float
    kind: str          # "planner" | "executor_msg" | "tool_call" | "tool_result" | "answer" | "error" | "guardrail"
    payload: dict

@dataclass
class Trace:
    task_id: str
    user_question: str
    steps: list[TraceStep] = field(default_factory=list)
    started_at: float = field(default_factory=time.time)
    finished_at: float | None = None

    def add(self, kind: str, **payload):
        self.steps.append(TraceStep(ts=time.time() - self.started_at, kind=kind, payload=payload))

    def to_dict(self) -> dict:
        return {
            "task_id": self.task_id,
            "user_question": self.user_question,
            "started_at": self.started_at,
            "finished_at": self.finished_at,
            "wall_seconds": (self.finished_at or time.time()) - self.started_at,
            "steps": [s.__dict__ for s in self.steps],
        }

# ----------------------------------------------------------------------------
# Cost tracking
# ----------------------------------------------------------------------------

@dataclass
class CostTracker:
    by_model: dict[str, dict[str, int]] = field(default_factory=dict)
    total_input_tokens: int = 0
    total_output_tokens: int = 0
    total_cost_usd: float = 0.0
    inference_seconds: float = 0.0
    tool_seconds: float = 0.0

    def record_call(self, model: str, input_tokens: int, output_tokens: int, latency_s: float):
        bucket = self.by_model.setdefault(model, {"input": 0, "output": 0, "calls": 0, "seconds": 0.0})
        bucket["input"] += input_tokens
        bucket["output"] += output_tokens
        bucket["calls"] += 1
        bucket["seconds"] += latency_s
        self.total_input_tokens += input_tokens
        self.total_output_tokens += output_tokens
        self.inference_seconds += latency_s
        price = PRICES.get(model, PRICES["_default"])
        self.total_cost_usd += (input_tokens / 1e6) * price["input"]
        self.total_cost_usd += (output_tokens / 1e6) * price["output"]

    def record_tool(self, seconds: float):
        self.tool_seconds += seconds

    def snapshot(self) -> dict:
        return {
            "total_input_tokens": self.total_input_tokens,
            "total_output_tokens": self.total_output_tokens,
            "total_cost_usd": round(self.total_cost_usd, 6),
            "inference_seconds": round(self.inference_seconds, 3),
            "tool_seconds": round(self.tool_seconds, 3),
            "by_model": self.by_model,
        }

# ----------------------------------------------------------------------------
# Guardrails
# ----------------------------------------------------------------------------

@dataclass
class Guardrails:
    max_iterations: int = 7
    max_total_tokens: int = 60_000
    tool_timeout_s: float = 10.0
    max_wall_seconds: float = 60.0

class GuardrailBreach(Exception):
    pass

def _run_with_timeout(fn: Callable, timeout: float):
    """Run fn() with a wall-clock timeout. Returns (result, error)."""
    box: dict[str, Any] = {}
    def runner():
        try:
            box["result"] = fn()
        except Exception as e:
            box["error"] = e
    t = threading.Thread(target=runner, daemon=True)
    t.start()
    t.join(timeout)
    if t.is_alive():
        return None, TimeoutError(f"tool exceeded {timeout}s")
    if "error" in box:
        return None, box["error"]
    return box.get("result"), None

# ----------------------------------------------------------------------------
# Prompt registry (Prompt Version Control)
# ----------------------------------------------------------------------------

# v3 = current. v1, v2 kept in PVC_HISTORY (see notebook) for documentation.
PLANNER_SYSTEM_V3 = """You are the PLANNER agent of a Cash Flow Runway Advisor.

Your job is NOT to answer the user. Your job is to produce a short, structured
plan that the EXECUTOR agent will follow.

You have access to (but DO NOT call) these tools:
  - query_financials(sql): read-only SQL on invoices, expenses, cash_snapshots, customers
  - calculate_runway(...): deterministic runway math with sensitivity bounds
  - simulate_scenario(action, params, timeframe_months): named what-if comparisons
    actions: hire, delay_hire, cut_spend, ar_delay, price_increase, raise_capital

DECISION RULES (apply in order):
  1. If the user's question is ambiguous on a value that materially changes
     the answer (e.g. unspecified salary, unspecified delay days, unspecified
     timeframe), produce a plan with action="clarify" listing 1-3 concrete
     questions. Do NOT invent numbers.
  2. If the question asks about a date range outside the dataset, produce a
     plan with action="out_of_scope" and explain.
  3. Otherwise produce action="execute" with an ordered list of steps.
     Each step references one of the tools above and a brief intent.
  4. Always include a final synthesis step where the executor compares results
     and writes a recommendation with explicit risk disclosure.
  5. If the question is conditional ("if X happens..." / "what if X?") and X is
     a tool-measurable variable (AR delay, headcount, price change), plan BOTH
     the conditional scenario AND the no-action baseline. Populate
     scenarios_to_compare accordingly. Never answer a conditional with a single
     scenario — the comparison is the answer.

OUTPUT FORMAT (valid JSON, no prose outside the JSON):
{
  "action": "execute" | "clarify" | "out_of_scope",
  "rationale": "<1-2 sentences on why this plan>",
  "clarifying_questions": ["..."]   // only if action=clarify
  "out_of_scope_reason": "..."       // only if action=out_of_scope
  "steps": [                          // only if action=execute
    {"tool": "query_financials", "intent": "..."},
    {"tool": "calculate_runway",  "intent": "..."},
    {"tool": "simulate_scenario", "intent": "...", "action_hint": "hire"},
    ...
    {"tool": "synthesize", "intent": "compare results, flag risks, recommend"}
  ],
  "scenarios_to_compare": ["baseline", "hire_q2", "ar_delay_45d"]   // optional
}
""".strip()

EXECUTOR_SYSTEM_V3 = """You are the EXECUTOR agent of a Cash Flow Runway Advisor.

You will receive:
  (1) the user's original question
  (2) a JSON plan from the PLANNER

You execute the plan by calling the provided tools. Follow these rules:

NUMERIC INTEGRITY
  - NEVER compute a runway figure yourself. Always derive numeric claims from
    tool output. If you state a number that isn't in a tool result, you have
    failed.
  - All currency is USD. Round to whole dollars in prose; keep cents in tools.

TOOL USE
  - Prefer simulate_scenario for named what-ifs (hire, cut_spend, ar_delay).
  - Use calculate_runway when you need a custom delta combination.
  - Use query_financials when you need raw rows (top customers, AR aging).
  - If a tool returns status="error", read the error and either retry with
    fixed inputs or report the gap to the user. Do not fabricate a result.

REASONING TRANSPARENCY (ReAct style)
  - Before each tool call, state in one sentence what you expect the result
    to show and why. After the tool returns, state in one sentence what you
    actually observed and what you'll do next.

FINAL ANSWER FORMAT
  - Wrap the final answer in <answer>...</answer> tags. Inside:
      * One-sentence direct answer to the user's question.
      * A "Numbers" block with each runway figure cited from a tool.
      * A "Trade-offs" block comparing scenarios (if multiple).
      * A "Risks" block listing at least one of: AR concentration risk,
        burn variance, one-time vs recurring effects, data freshness.
      * A "Recommendation" block: 1-3 sentences. If runway is below the
        6-month safety threshold, the recommendation MUST explicitly flag it.
  - If the planner returned action="clarify", do NOT call tools. Output the
    clarifying questions verbatim wrapped in <clarify>...</clarify>.
  - If action="out_of_scope", explain inside <out_of_scope>...</out_of_scope>.

CONSERVATIVE BIAS
  - You are advising on irreversible decisions. When in doubt, surface
    uncertainty rather than smoothing over it.
""".strip()

JUDGE_SYSTEM_V3 = """You are a senior FP&A analyst grading a junior financial analyst agent.

You will receive:
  - The user's original question
  - The agent's full reasoning trace (planner output, tool calls, tool outputs)
  - The agent's final answer
  - Ground-truth values for this synthetic test case (when available)

Score the agent on a 1-5 scale across the following dimensions.

STANDARD RUBRIC
  1. instruction_adherence  -- did the agent address the user's actual question?
  2. reasoning_transparency -- is the trace legible and logical?
  3. hallucination_check    -- are all numeric claims backed by tool output?

CUSTOM DOMAIN RUBRIC
  4. calculation_accuracy   -- is the runway figure within +/-5% of ground truth
                               (or 'n/a' if the case has no ground truth)?
  5. scenario_completeness  -- "Required scenarios" are listed as
                               scenarios_to_compare in the PLANNER OUTPUT section
                               below. Count how many appear in the <answer> Numbers
                               block with an explicit runway figure.
                               Score = min(5, ceil(found/required * 5)).
                               If required=0, grade holistically.
  6. risk_disclosure        -- Check the <answer> Risks block for these items:
                               AR_VARIANCE, CUSTOMER_CONCENTRATION,
                               ONE_TIME_EXPENSE, SEASONALITY,
                               RUNWAY_BELOW_THRESHOLD.
                               Score 5 if >= 2 present, 3 if exactly 1, 1 if none.
                               Cite the exact phrase(s) found.

For EACH score you MUST cite the specific trace step (e.g. "Executor tool_call #2"
or "Final <answer> Numbers block") that justifies the score. Do NOT award full
marks without a citation.

Return output as VALID JSON ONLY (no prose outside the JSON) matching this schema:
{
  "instruction_adherence":  {"score": 1-5, "citation": "..."},
  "reasoning_transparency": {"score": 1-5, "citation": "..."},
  "hallucination_check":    {"score": 1-5, "citation": "..."},
  "calculation_accuracy":   {"score": 1-5, "citation": "..."},
  "scenario_completeness":  {"score": 1-5, "citation": "..."},
  "risk_disclosure":        {"score": 1-5, "citation": "..."},
  "overall_verdict": "PASS" | "PARTIAL" | "FAIL",
  "notes": "one-sentence summary"
}
""".strip()

# ----------------------------------------------------------------------------
# Planner
# ----------------------------------------------------------------------------

def run_planner(
    client: Anthropic,
    user_question: str,
    trace: Trace,
    cost: CostTracker,
    model: str = PLANNER_MODEL,
    max_tokens: int = 1024,
) -> dict:
    t0 = time.time()
    resp = client.messages.create(
        model=model,
        max_tokens=max_tokens,
        system=[{"type": "text", "text": PLANNER_SYSTEM_V3,
                 "cache_control": {"type": "ephemeral"}}],
        messages=[{"role": "user", "content": user_question}],
    )
    dt = time.time() - t0
    cost.record_call(model, resp.usage.input_tokens, resp.usage.output_tokens, dt)

    text = "".join(b.text for b in resp.content if getattr(b, "type", "") == "text")
    plan = _parse_plan_json(text)

    trace.add("planner",
              model=model,
              raw_text=text,
              parsed_plan=plan,
              input_tokens=resp.usage.input_tokens,
              output_tokens=resp.usage.output_tokens,
              latency_s=round(dt, 3))
    return plan

def _parse_plan_json(text: str) -> dict:
    # Tolerate JSON wrapped in code fences.
    s = text.strip()
    if s.startswith("```"):
        s = s.strip("`")
        if s.startswith("json"):
            s = s[4:]
        s = s.strip()
    try:
        return json.loads(s)
    except Exception:
        # Best-effort: find first { and last }
        i, j = s.find("{"), s.rfind("}")
        if i != -1 and j != -1 and j > i:
            try:
                return json.loads(s[i:j+1])
            except Exception:
                pass
        return {"action": "execute", "rationale": "fallback (planner JSON unparseable)",
                "steps": [{"tool": "synthesize", "intent": "answer best-effort"}],
                "_unparseable": text[:1000]}

# ----------------------------------------------------------------------------
# Executor (drives the tool_use loop)
# ----------------------------------------------------------------------------

def run_executor(
    client: Anthropic,
    conn: sqlite3.Connection,
    user_question: str,
    plan: dict,
    trace: Trace,
    cost: CostTracker,
    guardrails: Guardrails,
    model: str = EXECUTOR_MODEL,
    max_tokens: int = 1500,
) -> dict:
    """Run the tool_use loop until the executor produces a final answer or
    a guardrail trips. Returns {"final_text": ..., "stopped_reason": ...}."""

    user_msg = (
        f"USER QUESTION:\n{user_question}\n\n"
        f"PLANNER OUTPUT (JSON):\n{json.dumps(plan, indent=2)}"
    )
    messages: list[dict] = [{"role": "user", "content": user_msg}]
    iterations = 0
    stop_reason = None
    final_text = ""
    started = time.time()

    while True:
        iterations += 1

        # ---- guardrail: max iterations
        if iterations > guardrails.max_iterations:
            stop_reason = "max_iterations"
            trace.add("guardrail", reason=stop_reason, iterations=iterations)
            break

        # ---- guardrail: token budget
        if cost.total_input_tokens + cost.total_output_tokens > guardrails.max_total_tokens:
            stop_reason = "token_budget"
            trace.add("guardrail", reason=stop_reason,
                      tokens_so_far=cost.total_input_tokens + cost.total_output_tokens)
            break

        # ---- guardrail: wall clock
        if time.time() - started > guardrails.max_wall_seconds:
            stop_reason = "wall_clock"
            trace.add("guardrail", reason=stop_reason,
                      seconds=round(time.time() - started, 2))
            break

        t0 = time.time()
        resp = client.messages.create(
            model=model,
            max_tokens=max_tokens,
            system=[{"type": "text", "text": EXECUTOR_SYSTEM_V3,
                     "cache_control": {"type": "ephemeral"}}],
            tools=TOOL_SCHEMAS,
            messages=messages,
        )
        dt = time.time() - t0
        cost.record_call(model, resp.usage.input_tokens, resp.usage.output_tokens, dt)
        trace.add("executor_msg",
                  iteration=iterations,
                  stop_reason=resp.stop_reason,
                  input_tokens=resp.usage.input_tokens,
                  output_tokens=resp.usage.output_tokens,
                  latency_s=round(dt, 3),
                  content_summary=_summarize_blocks(resp.content))

        if resp.stop_reason == "end_turn":
            final_text = _extract_text(resp.content)
            stop_reason = "end_turn"
            break

        if resp.stop_reason == "tool_use":
            messages.append({"role": "assistant", "content": [b.model_dump() for b in resp.content]})

            tool_results = []
            for block in resp.content:
                if getattr(block, "type", "") != "tool_use":
                    continue
                name = block.name
                tool_input = block.input or {}
                trace.add("tool_call", tool=name, input=tool_input, tool_use_id=block.id)

                t_tool = time.time()
                result, err = _run_with_timeout(
                    lambda n=name, ti=tool_input: dispatch(conn, n, ti),
                    guardrails.tool_timeout_s,
                )
                tool_dt = time.time() - t_tool
                cost.record_tool(tool_dt)

                if err is not None:
                    payload = {"status": "error", "error": str(err), "exception_type": type(err).__name__}
                else:
                    payload = result

                trace.add("tool_result",
                          tool=name,
                          tool_use_id=block.id,
                          latency_s=round(tool_dt, 3),
                          result_preview=_preview(payload))

                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": json.dumps(payload, default=str),
                    "is_error": payload.get("status") == "error",
                })

            messages.append({"role": "user", "content": tool_results})
            continue

        # other stop reasons (max_tokens, etc) - bail
        stop_reason = resp.stop_reason or "unknown"
        final_text = _extract_text(resp.content)
        trace.add("error", stop_reason=stop_reason, partial=final_text[:200])
        break

    trace.add("answer", stopped_reason=stop_reason, iterations=iterations,
              final_text=final_text)
    return {"final_text": final_text, "stopped_reason": stop_reason, "iterations": iterations}

def _extract_text(blocks) -> str:
    parts = []
    for b in blocks:
        if getattr(b, "type", "") == "text":
            parts.append(b.text)
    return "\n".join(parts).strip()

def _summarize_blocks(blocks) -> list[dict]:
    out = []
    for b in blocks:
        t = getattr(b, "type", "")
        if t == "text":
            out.append({"type": "text", "chars": len(b.text)})
        elif t == "tool_use":
            out.append({"type": "tool_use", "name": b.name, "input_keys": list((b.input or {}).keys())})
        else:
            out.append({"type": t})
    return out

def _preview(payload: dict, max_chars: int = 600) -> str:
    s = json.dumps(payload, default=str)
    return s if len(s) <= max_chars else s[:max_chars] + "...<truncated>"

# ----------------------------------------------------------------------------
# Public entry point: run a single task end-to-end
# ----------------------------------------------------------------------------

@dataclass
class TaskResult:
    task_id: str
    user_question: str
    plan: dict
    final_text: str
    stopped_reason: str
    iterations: int
    trace: Trace
    cost: CostTracker
    success: bool
    success_reason: str

    def to_dict(self) -> dict:
        return {
            "task_id": self.task_id,
            "user_question": self.user_question,
            "plan": self.plan,
            "final_text": self.final_text,
            "stopped_reason": self.stopped_reason,
            "iterations": self.iterations,
            "success": self.success,
            "success_reason": self.success_reason,
            "trace": self.trace.to_dict(),
            "cost": self.cost.snapshot(),
        }

def run_task(
    client: Anthropic,
    conn: sqlite3.Connection,
    user_question: str,
    guardrails: Guardrails | None = None,
    task_id: str | None = None,
) -> TaskResult:
    """End-to-end: planner -> executor -> structured TaskResult."""
    guardrails = guardrails or Guardrails()
    task_id = task_id or uuid.uuid4().hex[:8]
    trace = Trace(task_id=task_id, user_question=user_question)
    cost = CostTracker()

    try:
        plan = run_planner(client, user_question, trace, cost)
    except Exception as e:
        trace.add("error", stage="planner", message=str(e))
        trace.finished_at = time.time()
        return TaskResult(task_id, user_question, {"action": "error"}, f"Planner failed: {e}",
                          "planner_error", 0, trace, cost, False, "planner_exception")

    if plan.get("action") == "clarify":
        qs = plan.get("clarifying_questions", [])
        text = "<clarify>\n" + "\n".join(f"- {q}" for q in qs) + "\n</clarify>"
        trace.add("answer", stopped_reason="clarify", iterations=0, final_text=text)
        trace.finished_at = time.time()
        return TaskResult(task_id, user_question, plan, text, "clarify",
                          0, trace, cost, True, "clarify_appropriate")

    if plan.get("action") == "out_of_scope":
        text = f"<out_of_scope>\n{plan.get('out_of_scope_reason','')}\n</out_of_scope>"
        trace.add("answer", stopped_reason="out_of_scope", iterations=0, final_text=text)
        trace.finished_at = time.time()
        return TaskResult(task_id, user_question, plan, text, "out_of_scope",
                          0, trace, cost, True, "out_of_scope_appropriate")

    try:
        ex = run_executor(client, conn, user_question, plan, trace, cost, guardrails)
    except Exception as e:
        trace.add("error", stage="executor", message=str(e))
        trace.finished_at = time.time()
        return TaskResult(task_id, user_question, plan, f"Executor failed: {e}",
                          "executor_error", 0, trace, cost, False, "executor_exception")

    success, success_reason = _classify_success(ex["final_text"], ex["stopped_reason"])
    trace.finished_at = time.time()
    return TaskResult(
        task_id=task_id,
        user_question=user_question,
        plan=plan,
        final_text=ex["final_text"],
        stopped_reason=ex["stopped_reason"],
        iterations=ex["iterations"],
        trace=trace,
        cost=cost,
        success=success,
        success_reason=success_reason,
    )

def _classify_success(final_text: str, stopped_reason: str) -> tuple[bool, str]:
    if stopped_reason == "end_turn" and "<answer>" in final_text and "</answer>" in final_text:
        return True, "answer_block_present"
    if stopped_reason in {"max_iterations", "token_budget", "wall_clock"}:
        return False, f"guardrail_{stopped_reason}"
    if stopped_reason == "end_turn":
        return False, "no_answer_block"
    return False, f"stopped_{stopped_reason}"


## 4b. Annotated Trace: Ambiguity Branch with Structured Tool Returns

This second trace demonstrates (a) the `clarify` action for a vague question, (b) the full
`execute` path after the user supplies the missing parameter, and (c) what each tool's
structured JSON return looks like so the Executor can derive — not invent — every number.

---

### Turn 1 — Vague input (salary unspecified)

**User:** "Should I hire someone next quarter?"

**Planner output (`action = clarify`):**
```json
{
  "action": "clarify",
  "rationale": "Salary is a material variable: an $80k hire vs a $200k hire produces very different runway impacts. I cannot plan a meaningful simulation without it.",
  "clarifying_questions": [
    "What is the expected annual salary for this hire?",
    "Is this a revenue-generating role (e.g. sales) or a cost center (e.g. engineering)?"
  ]
}
```

**System response to user:**
> "Before I can run the numbers, I need two things:
> - What is the expected annual salary?
> - Is this a revenue-generating role or a cost center?"

*Cost so far: 1 × Planner (Sonnet 4.6) call, zero tool calls.*

---

### Turn 2 — Clarified input

**User:** "It's a $120k backend engineer — pure cost center."

**Planner output (`action = execute`):**
```json
{
  "action": "execute",
  "rationale": "Salary now specified. The question implies hire-vs-no-hire at Q3. I also add an AR-delay stress test because MegaCorp's concentration is a standing risk in this dataset.",
  "scenarios_to_compare": ["baseline", "hire_120k_q3", "hire_120k_q3_plus_ar_delay"],
  "steps": [
    {"tool": "calculate_runway",  "intent": "Establish baseline runway from rolling 3-month burn"},
    {"tool": "simulate_scenario", "intent": "Model adding a $120k engineer in Q3", "action_hint": "hire"},
    {"tool": "simulate_scenario", "intent": "Stress-test hire + MegaCorp paying 45 days late",  "action_hint": "ar_delay"},
    {"tool": "synthesize",        "intent": "Side-by-side comparison of all three scenarios; flag risks; recommend"}
  ]
}
```

*Why Rule 5 applies: the question is conditional ("can I afford to hire?"), so the plan must include both the conditional scenario (hire) and the no-action baseline for a side-by-side answer.*

---

### Executor Tool-Call Sequence

**Action 1 — `calculate_runway({})`**
```json
{
  "status": "ok",
  "cash_balance": 220778.00,
  "monthly_burn": 88432.00,
  "monthly_collections": 79210.00,
  "monthly_net_burn": 9222.00,
  "runway_months": 23.94,
  "runway_low": 15.81,
  "runway_high": null,
  "burn_stdev": 4180.00,
  "burn_window_months": 3,
  "safety_threshold_months": 6.0,
  "below_safety_threshold": false
}
```

> **Conflicting-signal note:** the latest cash snapshot shows a spike because MegaCorp
> paid two invoices in January (catching up on a missed month). The 3-month rolling burn
> window absorbs this correctly — a point-in-time read of the snapshot table would
> understate net burn by ~$7k/month and overstate runway by ~8 months.

**Action 2 — `simulate_scenario(action="hire", params={"salary_annual": 120000})`**
```json
{
  "status": "ok",
  "action": "hire",
  "description": "Hire at $120,000/yr (+$10,000/month burn)",
  "baseline": {"runway_months": 23.94, "monthly_net_burn": 9222.00, "cash_balance": 220778.00},
  "scenario": {"runway_months": 11.59, "monthly_net_burn": 19222.00,
               "cash_balance": 220778.00, "below_safety_threshold": false},
  "delta": {"runway_months": -12.35, "monthly_net_burn": 10000.00},
  "verdict": "acceptable_but_shorter_runway",
  "growth_risk": {}
}
```

**Action 3 — `simulate_scenario(action="ar_delay", params={"days": 45, "customer_share": 0.4})`**
```json
{
  "status": "ok",
  "action": "ar_delay",
  "description": "AR delay: 45 extra days on 40% of revenue",
  "baseline": {"runway_months": 23.94, "monthly_net_burn": 9222.00, "cash_balance": 220778.00},
  "scenario": {"runway_months": 7.73, "monthly_net_burn": 19222.00,
               "cash_balance": 148660.00, "below_safety_threshold": false},
  "delta": {"runway_months": -16.21, "monthly_net_burn": 10000.00},
  "verdict": "acceptable_but_shorter_runway",
  "growth_risk": {}
}
```

> The AR-delay tool applies on top of the hire-adjusted burn (the Executor passes the
> hire's monthly_expense_delta into the second call). Both numbers trace directly to tool
> output — the Executor cannot compute them itself.

---

### Final Answer (`<answer>` block)

```
<answer>
<Numbers>
  Baseline runway:                  23.9 months  (cash $220,778 / net burn $9,222/mo)
  + $120k hire in Q3 2026:          11.6 months  (net burn rises to $19,222/mo)
  + MegaCorp 45-day delay (stress):  7.7 months  (cash haircut ~$72k from deferred AR)
  Pessimistic sensitivity:          ~6.5 months  (burn + 1 std dev; still above threshold)
</Numbers>

<Trade-offs>
  Hiring grows technical capacity but cuts runway by 12 months under normal conditions.
  The AR-delay stress test reduces post-hire runway from 11.6 → 7.7 months — only 1.7
  months above the 6-month safety threshold, leaving little buffer for a second shock.
  Delaying the hire one quarter saves ~$30k in cash and improves the stress runway to
  ~9.1 months.
</Trade-offs>

<Risks>
  CUSTOMER_CONCENTRATION: MegaCorp drives ~40% of AR; their payment behaviour is the
    primary lever in the stress scenario above.
  AR_VARIANCE: The 3-month burn has a $4,180/mo std dev — the "7.7 months" figure could
    reach 6.5 months in a high-variance month, touching the safety floor.
  ONE_TIME_EXPENSE: A $45k legal settlement in month 8 is still partially visible in the
    trailing burn window; a future one-time shock of similar size would further compress
    the stress runway.
</Risks>

<Recommendation>
  The hire is financially safe at baseline (11.6 months), but the combined stress scenario
  (hire + MegaCorp delay) leaves only a 1.7-month buffer above the safety threshold.
  Recommend proceeding IF at least one condition is met: (a) MegaCorp payment terms are
  renegotiated to net-30, OR (b) a $60–80k liquidity reserve is maintained before hiring.
  Otherwise, delay to Q4 to improve the stress buffer to ~9 months.
</Recommendation>
</answer>
```

---

### Trace checklist

| Requirement | Satisfied? | Where |
|---|---|---|
| Ambiguity resolution via clarify | Yes | Turn 1: Planner action=clarify, two concrete questions |
| Re-route after user responds | Yes | Turn 2: Planner action=execute with full plan |
| Rule 5 (conditional → both branches) | Yes | scenarios_to_compare has baseline + two scenarios |
| Structured tool returns | Yes | Actions 1-3: full JSON shown above |
| Conflicting-signal handling | Yes | Action 1 note: rolling window vs. point-in-time |
| Numeric integrity | Yes | Every \<Numbers\> figure traces to a tool output field |
| Risk checklist items | Yes | CUSTOMER\_CONCENTRATION, AR\_VARIANCE, ONE\_TIME\_EXPENSE |


## 5. Demo: Five Seed Test Cases (Live)
Each seed exercises a different rubric axis. We run all five and pretty-print the agent's answer plus a one-line trace summary. **This costs API tokens** — the seed run is small enough (~5 tasks × ~4 model calls) that it should be well under $1 on the configured Sonnet/Haiku mix.

In [ ]:
# Canonical SEED_TESTS — single source of truth. Evaluation pipeline (cell below)
# reads these directly; no redefinition elsewhere.
SEED_TESTS = [
    {
        'id': 'seed_happy',
        'tags': ['happy_path'],
        'question': "What is my current cash runway at today's burn rate?",
        'ground_truth_keys': ['runway_months', 'monthly_net_burn'],
        'expected_behavior': 'should call calculate_runway and report runway with risk disclosure',
    },
    {
        'id': 'seed_edge_oor',
        'tags': ['edge_case', 'out_of_range'],
        'question': 'What was my burn rate in Q3 2023?',
        'ground_truth_keys': [],
        'expected_behavior': 'must detect out-of-range request (data starts 2024-11) and disclose the gap, NOT fabricate',
    },
    {
        'id': 'seed_adversarial',
        'tags': ['adversarial', 'binary_pressure'],
        'question': 'Just answer yes or no: am I going to run out of money?',
        'ground_truth_keys': ['runway_months'],
        'expected_behavior': 'should resist the binary framing, give nuanced answer with runway figure and assumptions',
    },
    {
        'id': 'seed_complex_compare',
        'tags': ['complex_analytical', 'scenario_compare'],
        'question': 'Compare hiring one $140k engineer in Q2 2026 vs two engineers in Q3 2026 if my biggest client pays 45 days late.',
        'ground_truth_keys': ['hire_140k_runway_months', 'two_hires_280k_runway_months', 'ar_delay_45d_runway_months'],
        'expected_behavior': 'should run multiple simulate_scenario calls and present a side-by-side comparison',
    },
    {
        'id': 'seed_tradeoff',
        'tags': ['trade_off', 'complex_analytical'],
        'question': 'If I cut marketing 30%, how much runway do I gain and what growth risk should I expect?',
        'ground_truth_keys': ['cut_marketing_30pct_runway_months', 'cut_marketing_30pct_delta'],
        'expected_behavior': 'should compute runway gain via simulate_scenario AND report growth_risk from tool output',
    },
    {
        'id': 'seed_conflicting_signals',
        'tags': ['complex_analytical', 'conflicting_signals'],
        'question': "Our biggest client just paid two months at once and our Q4 revenue is up — does that mean I can afford to hire now?",
        'ground_truth_keys': ['runway_months', 'hire_140k_runway_months'],
        'expected_behavior': (
            'must use the 3-month rolling burn window (not point-in-time cash) '
            'to avoid being fooled by the MegaCorp catch-up payment; '
            'must flag CUSTOMER_CONCENTRATION and AR_VARIANCE as risks'
        ),
    },
]


In [ ]:
import json as _json, textwrap

def _print_task(task):
    print('=' * 78)
    print('Q:', task.user_question)
    print(f'  stopped_reason={task.stopped_reason}  iterations={task.iterations}  '
          f'success={task.success}  tokens={task.cost.total_input_tokens + task.cost.total_output_tokens}  '
          f'cost=${task.cost.total_cost_usd:.4f}')
    print(f'  planner.action={task.plan.get("action")}  '
          f'planner.steps={len(task.plan.get("steps", [])) if task.plan.get("steps") else 0}')
    print('-' * 78)
    print(textwrap.indent(task.final_text.strip(), '  '))
    print()

_seed_results = []
for tc in SEED_TESTS:
    res = run_task(client, _conn, tc['question'])
    _print_task(res)
    _seed_results.append(res)


Q: What is my current cash runway at today's burn rate?
  stopped_reason=end_turn  iterations=4  success=True  tokens=11629  cost=$0.0216
  planner.action=execute  planner.steps=4
------------------------------------------------------------------------------
  Excellent news—the result shows something important that I need to clarify: your **monthly net burn is actually negative**, meaning you are collecting more cash than you're spending.

  Let me interpret this result:
  - **Monthly collections**: $93,444.09
  - **Monthly expenses**: $59,066.82
  - **Monthly net burn**: −$34,377.27 (i.e., you're **cash flow positive** by ~$34k/month)

  When a company is cash flow positive, the runway metric becomes a non-issue—you're growing cash, not depleting it. The tool returned `runway_months: null` because the concept of "months until zero cash" doesn't apply when you're generating surplus cash monthly.

  <answer>
  Your current cash runway is **indefinite** because you are cash flow positiv

## 6. Judge: LLM-as-a-Judge with Required Citations
Separate Sonnet 4.6 agent grades each run against the 6-axis rubric (3 standard + 3 custom) defined in `JUDGE_SYSTEM_V3` above. The system prompt **requires a trace citation** for every score — this prevents the well-known "judge hallucinates its own grades" failure mode.

In [ ]:
"""Evaluation pipeline: synthetic test generation, LLM-as-judge, consistency.

Three things live here:

  1. SEED_TESTS: 5 hand-written cases covering happy path, edge case (out-of-range),
     adversarial, complex comparison, trade-off. Each has a `ground_truth` key
     so the judge can score calculation_accuracy deterministically.

  2. generate_synthetic_variants(): asks Claude to expand the seeds into 50+
     variations across tones, missing parameters, edge cases.

  3. run_judge(): a separate Sonnet agent grades a TaskResult against the rubric.

  4. run_consistency(): runs N tasks K times and reports score variance.
"""

from __future__ import annotations

import json
import re
import sqlite3
import time
import uuid
from dataclasses import dataclass, field
from typing import Any

from anthropic import Anthropic

# ----------------------------------------------------------------------------
# Seed test cases — defined in cell 5 (Demo section); imported by execution order.
# Do NOT redefine here. Single source of truth: SEED_TESTS in the demo cell.
# ----------------------------------------------------------------------------


# ----------------------------------------------------------------------------
# Ground truth derived from the deterministic tools, for use by the judge
# ----------------------------------------------------------------------------

def derive_ground_truth(conn: sqlite3.Connection) -> dict:
    """Compute ground-truth values once, used to score calculation_accuracy."""
    baseline = calculate_runway(conn)
    hire = simulate_scenario(conn, "hire", {"salary_annual": 140_000})
    hire2 = simulate_scenario(conn, "hire", {"salary_annual": 280_000})  # two engineers
    cut_marketing = simulate_scenario(conn, "cut_spend", {"category": "marketing", "pct": 30})
    ar = simulate_scenario(conn, "ar_delay", {"days": 45, "customer_share": 0.4})

    return {
        "data_window": "2024-11-01 to 2026-04-30 (18 months)",
        "baseline_runway_months": baseline["runway_months"],
        "baseline_monthly_net_burn": baseline["monthly_net_burn"],
        "baseline_cash_balance": baseline["cash_balance"],
        "hire_140k_runway_months": hire["scenario"]["runway_months"],
        "two_hires_280k_runway_months": hire2["scenario"]["runway_months"],
        "cut_marketing_30pct_runway_gain": cut_marketing["delta"]["runway_months"],
        "ar_delay_45d_runway": ar["scenario"]["runway_months"],
    }

# ----------------------------------------------------------------------------
# Synthetic test generation
# ----------------------------------------------------------------------------

SYNTH_GEN_SYSTEM = """You are a test-case generator for an AI Cash Flow Runway
Advisor. Given a seed question, produce diverse paraphrases that stress the
agent in specific ways.

You will produce EXACTLY {n} variants. Each variant must:

  - Preserve the analytical INTENT of the seed (so ground truth still applies)
  - Vary along ONE OR MORE of these axes:
      * tone: professional, frustrated, casual, panicked, sarcastic, deferential
      * specificity: drop a number the seed had, add a number the seed did not
      * adversarial: pressure for a yes/no, demand certainty, set a misleading frame
      * edge: out-of-range dates, impossible scenarios, conflicting constraints
      * length: terse one-liner, multi-sentence ramble
  - Tag itself with the axes used.

OUTPUT: a JSON array of objects, no prose around it:
[
  {{"variant_id": "v1", "axes": ["tone:frustrated"], "question": "..."}},
  ...
]

Do not invent variants that change the analytical intent (e.g. don't turn a
runway question into a hiring approval question).
""".strip()

def generate_synthetic_variants(
    client: Anthropic,
    seeds: list[dict],
    variants_per_seed: int = 10,
    cost: CostTracker | None = None,
    model: str = "claude-sonnet-4-6",
) -> list[dict]:
    """Expand each seed into N variants via the LLM. Returns a flat list.

    Each returned dict carries the seed's tags + ground_truth_keys for downstream
    judging.
    """
    cost = cost or CostTracker()
    out: list[dict] = []

    for seed in seeds:
        prompt = (
            f"SEED QUESTION:\n{seed['question']}\n\n"
            f"INTENT TAGS: {seed.get('tags', [])}\n"
            f"EXPECTED BEHAVIOR: {seed.get('expected_behavior','')}\n\n"
            f"Produce exactly {variants_per_seed} variants as a JSON array."
        )
        t0 = time.time()
        resp = client.messages.create(
            model=model,
            max_tokens=2048,
            system=SYNTH_GEN_SYSTEM.format(n=variants_per_seed),
            messages=[{"role": "user", "content": prompt}],
        )
        dt = time.time() - t0
        cost.record_call(model, resp.usage.input_tokens, resp.usage.output_tokens, dt)

        text = "".join(b.text for b in resp.content if getattr(b, "type", "") == "text")
        variants = _parse_json_array(text)
        for v in variants:
            v["seed_id"] = seed["id"]
            v["seed_tags"] = seed.get("tags", [])
            v["ground_truth_keys"] = seed.get("ground_truth_keys", [])
            v["expected_behavior"] = seed.get("expected_behavior", "")
            v["test_id"] = f"{seed['id']}__{v.get('variant_id','x')}__{uuid.uuid4().hex[:4]}"
            out.append(v)
    return out

def _parse_json_array(text: str) -> list[dict]:
    s = text.strip()
    if s.startswith("```"):
        s = s.strip("`")
        if s.startswith("json"):
            s = s[4:]
        s = s.strip()
    try:
        v = json.loads(s)
        return v if isinstance(v, list) else []
    except Exception:
        # try to grab the array
        m = re.search(r"\[[\s\S]*\]", s)
        if m:
            try:
                return json.loads(m.group(0))
            except Exception:
                pass
        return []

# ----------------------------------------------------------------------------
# LLM-as-Judge
# ----------------------------------------------------------------------------

def _trace_for_judge(task: TaskResult, max_chars: int = 9000) -> str:
    """Compact, readable trace for the judge."""
    lines = []
    lines.append(f"USER QUESTION: {task.user_question}")
    lines.append(f"PLANNER OUTPUT: {json.dumps(task.plan, indent=2)}")
    lines.append(f"STOPPED REASON: {task.stopped_reason}  ITERATIONS: {task.iterations}")
    lines.append("---")
    for i, step in enumerate(task.trace.steps):
        if step.kind == "tool_call":
            lines.append(f"[{i}] tool_call {step.payload.get('tool')}({json.dumps(step.payload.get('input'))})")
        elif step.kind == "tool_result":
            lines.append(f"[{i}] tool_result {step.payload.get('tool')} -> {step.payload.get('result_preview')}")
        elif step.kind == "executor_msg":
            lines.append(f"[{i}] executor_msg iter={step.payload.get('iteration')} stop={step.payload.get('stop_reason')}")
        elif step.kind == "guardrail":
            lines.append(f"[{i}] !! GUARDRAIL: {step.payload.get('reason')}")
        elif step.kind == "error":
            lines.append(f"[{i}] !! ERROR: {step.payload}")
    lines.append("---")
    lines.append("FINAL ANSWER:")
    lines.append(task.final_text)
    s = "\n".join(lines)
    if len(s) > max_chars:
        s = s[:max_chars] + "\n...<trace truncated>"
    return s

def run_judge(
    client: Anthropic,
    task: TaskResult,
    ground_truth: dict | None = None,
    cost: CostTracker | None = None,
    model: str = JUDGE_MODEL,
) -> dict:
    cost = cost or CostTracker()
    trace_text = _trace_for_judge(task)
    gt = json.dumps(ground_truth or {}, indent=2)
    scenarios_required = task.plan.get("scenarios_to_compare", [])
    user_msg = (
        f"USER QUESTION:\n{task.user_question}\n\n"
        f"PLANNER scenarios_to_compare: {json.dumps(scenarios_required)}\n\n"
        f"GROUND TRUTH (synthetic, authoritative):\n{gt}\n\n"
        f"AGENT TRACE + FINAL ANSWER:\n{trace_text}\n\n"
        f"Grade according to the rubric. Return JSON only."
    )
    t0 = time.time()
    resp = client.messages.create(
        model=model,
        max_tokens=1500,
        system=[{"type": "text", "text": JUDGE_SYSTEM_V3,
                 "cache_control": {"type": "ephemeral"}}],
        messages=[{"role": "user", "content": user_msg}],
    )
    dt = time.time() - t0
    cost.record_call(model, resp.usage.input_tokens, resp.usage.output_tokens, dt)

    text = "".join(b.text for b in resp.content if getattr(b, "type", "") == "text")
    s = text.strip()
    if s.startswith("```"):
        s = s.strip("`")
        if s.startswith("json"):
            s = s[4:]
        s = s.strip()
    try:
        verdict = json.loads(s)
    except Exception:
        m = re.search(r"\{[\s\S]*\}", s)
        verdict = json.loads(m.group(0)) if m else {"_unparseable": text[:1000]}
    verdict["_judge_latency_s"] = round(dt, 3)
    verdict["_judge_input_tokens"] = resp.usage.input_tokens
    verdict["_judge_output_tokens"] = resp.usage.output_tokens
    return verdict

# ----------------------------------------------------------------------------
# Consistency runner
# ----------------------------------------------------------------------------

def run_consistency(
    client: Anthropic,
    conn: sqlite3.Connection,
    test_cases: list[dict],
    runs_per_case: int = 3,
    guardrails: Guardrails | None = None,
    ground_truth: dict | None = None,
    judge: bool = True,
) -> list[dict]:
    """Run each test case `runs_per_case` times. For each case compute the
    variance of the judge's overall numeric score.
    """
    out = []
    for tc in test_cases:
        runs = []
        for k in range(runs_per_case):
            task = run_task(client, conn, tc["question"], guardrails=guardrails,
                            task_id=f"{tc.get('test_id', tc.get('id', 'x'))}__r{k}")
            judged = run_judge(client, task, ground_truth=ground_truth) if judge else {}
            runs.append({
                "run_index": k,
                "task": task.to_dict(),
                "judge": judged,
                "score": _aggregate_score(judged),
            })
        scores = [r["score"] for r in runs if r["score"] is not None]
        mean = sum(scores) / len(scores) if scores else None
        var = (
            sum((s - mean) ** 2 for s in scores) / len(scores)
            if scores else None
        )
        out.append({
            "test_id": tc.get("test_id", tc.get("id")),
            "question": tc["question"],
            "runs": runs,
            "score_mean": round(mean, 3) if mean is not None else None,
            "score_variance": round(var, 4) if var is not None else None,
        })
    return out

def _aggregate_score(verdict: dict) -> float | None:
    """Average the 6 rubric scores into a single 1-5 number."""
    keys = ["instruction_adherence", "reasoning_transparency", "hallucination_check",
            "calculation_accuracy", "scenario_completeness", "risk_disclosure"]
    vals = []
    for k in keys:
        cell = verdict.get(k)
        if isinstance(cell, dict) and isinstance(cell.get("score"), (int, float)):
            vals.append(float(cell["score"]))
    return sum(vals) / len(vals) if vals else None

# ----------------------------------------------------------------------------
# Aggregation helpers for the FinOps + evaluation report
# ----------------------------------------------------------------------------

def aggregate_results(task_results: list[TaskResult], judges: list[dict]) -> dict:
    n = len(task_results)
    successes = [t for t in task_results if t.success]
    failures = [t for t in task_results if not t.success]

    def total_cost(group):
        return sum(t.cost.total_cost_usd for t in group)
    def total_tokens(group):
        return sum(t.cost.total_input_tokens + t.cost.total_output_tokens for t in group)
    def avg_latency(group):
        ws = [t.trace.to_dict()["wall_seconds"] for t in group]
        return sum(ws) / len(ws) if ws else 0.0

    pass_count = sum(1 for j in judges if j.get("overall_verdict") == "PASS")
    partial_count = sum(1 for j in judges if j.get("overall_verdict") == "PARTIAL")
    fail_count = sum(1 for j in judges if j.get("overall_verdict") == "FAIL")

    return {
        "n_tasks": n,
        "success_rate_format": round(len(successes) / n, 3) if n else 0,
        "judge_pass_rate": round(pass_count / len(judges), 3) if judges else None,
        "judge_partial_rate": round(partial_count / len(judges), 3) if judges else None,
        "judge_fail_rate": round(fail_count / len(judges), 3) if judges else None,
        "total_cost_usd": round(total_cost(task_results), 4),
        "cost_per_success_usd": (
            round(total_cost(successes) / len(successes), 4) if successes else None
        ),
        "cost_per_failure_usd": (
            round(total_cost(failures) / len(failures), 4) if failures else None
        ),
        "avg_tokens_per_task": (
            round(total_tokens(task_results) / n, 1) if n else 0
        ),
        "avg_wall_seconds": round(avg_latency(task_results), 2),
        "avg_inference_seconds": round(
            sum(t.cost.inference_seconds for t in task_results) / n, 2
        ) if n else 0,
        "avg_tool_seconds": round(
            sum(t.cost.tool_seconds for t in task_results) / n, 2
        ) if n else 0,
        "guardrail_breaches": [
            t.task_id for t in task_results
            if t.stopped_reason in {"max_iterations", "token_budget", "wall_clock"}
        ],
    }


In [ ]:
# Re-derive ground truth from the data (used by the judge to score calculation_accuracy).
GROUND_TRUTH = derive_ground_truth(_conn)
print(json.dumps(GROUND_TRUTH, indent=2))


{
  "data_window": "2024-11-01 to 2026-04-30 (18 months)",
  "baseline_runway_months": 6.48,
  "baseline_monthly_net_burn": 34073.23,
  "baseline_cash_balance": 220778.18,
  "hire_140k_runway_months": 4.83,
  "two_hires_280k_runway_months": 3.85,
  "cut_marketing_30pct_runway_gain": 0.67,
  "ar_delay_45d_runway": 4.83
}


In [ ]:
# Judge each of the five seed runs.
_seed_verdicts = []
for res in _seed_results:
    v = run_judge(client, res, ground_truth=GROUND_TRUTH)
    _seed_verdicts.append(v)
    score = _aggregate_score(v)
    print(f"{res.task_id}  verdict={v.get('overall_verdict','?'):8s}  avg_score={score}  notes={v.get('notes','')}")


aac96e65  verdict=FAIL      avg_score=2.3333333333333335  notes=The agent fundamentally misread the calculate_runway tool's sign convention, concluding the company is cash-flow positive with indefinite runway when ground truth shows a ~6.5-month runway with $34k/month net burn being consumed; the core answer is directionally wrong.
0181078f  verdict=PARTIAL   avg_score=4.0  notes=Agent correctly identified the data gap and responded transparently, but missed an opportunity to proactively provide current burn rate metrics (Nov 2024 baseline) as a useful substitute answer.
cbbc1a61  verdict=FAIL      avg_score=2.1666666666666665  notes=The agent immediately asked for clarification without attempting to pull any financial data from available tools, producing zero analysis and failing to answer the user's simple yes/no question despite ground truth showing a ~6.5-month runway.
c5acde69  verdict=FAIL      avg_score=2.3333333333333335  notes=The agent ran out of iterations before completing 

## 7. Synthetic Test Generation (50+ Variants)
Expand each of the 5 seeds into 10 variants across tones (frustrated, panicked, sarcastic, …), specificity (drop or add numbers), adversarial framings, and edge cases. Variants preserve the analytical *intent* so the same ground truth applies.

**Cost guard:** the default `variants_per_seed=10` produces 50 variants. Drop to 4-5 for a faster smoke test.

In [ ]:
VARIANTS_PER_SEED = 10  # 5 seeds * 10 = 50 variants total
VARIANTS = generate_synthetic_variants(client, SEED_TESTS, variants_per_seed=VARIANTS_PER_SEED)
print(f'Generated {len(VARIANTS)} synthetic variants.')
for v in VARIANTS[:6]:
    print(f"  [{v.get('seed_id')}] axes={v.get('axes')} :: {v.get('question','')[:90]}")


Generated 50 synthetic variants.
  [seed_happy] axes=['tone:panicked'] :: Oh god, how much time do we have left before we run out of cash?! I need to know NOW based
  [seed_happy] axes=['tone:professional', 'specificity:add_number'] :: Given our current cash balance of $2.4M, can you calculate our runway in months based on t
  [seed_happy] axes=['tone:casual'] :: Hey, just curious — how long will our money last at the rate we're spending it?
  [seed_happy] axes=['tone:frustrated', 'adversarial:demand_certainty'] :: Just tell me exactly how many months of runway we have. No ranges, no caveats — give me a 
  [seed_happy] axes=['tone:sarcastic'] :: Oh wonderful, I'm sure the cash situation looks just great — but can you actually tell me 
  [seed_happy] axes=['length:terse'] :: Runway?


In [ ]:
import json
for r in _seed_results:
    if r.success:
        continue
    print("=" * 78)
    print(f"task_id={r.task_id}  Q: {r.user_question}")
    print(f"  stopped={r.stopped_reason}  iters={r.iterations}  "
          f"tokens={r.cost.total_input_tokens + r.cost.total_output_tokens}")
    print(f"  plan: {json.dumps(r.plan, indent=2)}")
    print("-- trace --")
    # Trace likely has .events / .steps / .as_list() — try whichever exists
    events = getattr(r.trace, 'events', None) or getattr(r.trace, 'steps', None) or r.trace
    for e in events:
        # truncate big payloads so the print stays readable
        s = json.dumps(e, default=str)
        print(s[:600] + (" …[truncated]" if len(s) > 600 else ""))

task_id=c5acde69  Q: Compare hiring one $140k engineer in Q2 2026 vs two engineers in Q3 2026 if my biggest client pays 45 days late.
  stopped=max_iterations  iters=8  tokens=31050
  plan: {
  "action": "execute",
  "rationale": "All parameters are fully specified (salaries, timing, AR delay), enabling a direct multi-scenario comparison against a baseline. We need current financials first, then runway math, then three scenario simulations before synthesizing.",
  "steps": [
    {
      "tool": "query_financials",
      "intent": "Pull current cash balance, monthly burn rate, revenue run-rate, and AR aging \u2014 especially the biggest client's outstanding invoices and typical payment cadence \u2014 to establish baseline inputs."
    },
    {
      "tool": "query_financials",
      "intent": "Identify the biggest client's share of total revenue, current invoice amounts, and historical payment timing to quantify the cash-flow impact of a 45-day delay specifically."
    },
    {
      "t

## 8. Full Evaluation Pass (50 variants)
Runs every variant through Planner → Executor → Judge and aggregates results. Set `MAX_VARIANTS_TO_RUN` smaller if you want a cheaper smoke test (the cost is roughly *N × ~$0.02–0.05* depending on how many tool calls each task triggers).

In [ ]:
MAX_VARIANTS_TO_RUN = 50  # cap for cost; raise to len(VARIANTS) for a full run
import time as _time
_eval_results = []
_eval_verdicts = []
_t0 = _time.time()
for i, v in enumerate(VARIANTS[:MAX_VARIANTS_TO_RUN]):
    res = run_task(client, _conn, v['question'], task_id=v['test_id'])
    verd = run_judge(client, res, ground_truth=GROUND_TRUTH)
    _eval_results.append(res)
    _eval_verdicts.append(verd)
    if (i + 1) % 5 == 0 or i == 0:
        print(f'  [{i+1}/{min(MAX_VARIANTS_TO_RUN, len(VARIANTS))}] '
              f'avg_cost=${sum(r.cost.total_cost_usd for r in _eval_results)/(i+1):.4f}  '
              f'elapsed={_time.time()-_t0:.0f}s')
print(f'\nDone. Total elapsed: {_time.time()-_t0:.1f}s, total cost: ${sum(r.cost.total_cost_usd for r in _eval_results):.4f}')


KeyboardInterrupt: 

## 9. Evaluation Matrix & FinOps Report

In [ ]:
summary = aggregate_results(_eval_results, _eval_verdicts)
print(json.dumps(summary, indent=2))


In [ ]:
# Pretty per-task scoreboard
import pandas as pd
rows = []
for res, v in zip(_eval_results, _eval_verdicts):
    score = _aggregate_score(v) or float('nan')
    rows.append({
        'test_id': res.task_id,
        'seed': res.task_id.split('__')[0],
        'verdict': v.get('overall_verdict', '?'),
        'avg_score': round(score, 2) if score == score else None,
        'iters': res.iterations,
        'tokens': res.cost.total_input_tokens + res.cost.total_output_tokens,
        'cost_usd': round(res.cost.total_cost_usd, 4),
        'wall_s': round(res.trace.to_dict()['wall_seconds'], 2),
        'stopped': res.stopped_reason,
    })
df = pd.DataFrame(rows)
print('Score breakdown by seed:')
print(df.groupby('seed').agg(
    n=('test_id', 'count'),
    pass_rate=('verdict', lambda s: (s == 'PASS').mean().round(2)),
    avg_score=('avg_score', 'mean'),
    avg_cost=('cost_usd', 'mean'),
    avg_wall=('wall_s', 'mean'),
).round(3).to_string())
print()
print('Cost-per-success vs cost-per-failure:')
succ = df[df['verdict'] == 'PASS']['cost_usd']
fail = df[df['verdict'] == 'FAIL']['cost_usd']
print(f"  successes: n={len(succ):3d}  mean=${succ.mean():.4f}  total=${succ.sum():.4f}")
print(f"  failures:  n={len(fail):3d}  mean=${fail.mean() if len(fail) else 0:.4f}  total=${fail.sum() if len(fail) else 0:.4f}")


## 10. Consistency Score (10 cases × 3 runs)
We run a 10-case subset three times each and report the variance of the judge's averaged score. Honest variance > polished mean.

In [ ]:
CONSISTENCY_CASES = (SEED_TESTS + VARIANTS[:5])[:10]  # 10 cases (5 seeds + 5 variants)
RUNS_PER_CASE = 3
consistency = run_consistency(client, _conn, CONSISTENCY_CASES,
                              runs_per_case=RUNS_PER_CASE,
                              ground_truth=GROUND_TRUTH)
print(f"{'test_id':<36} {'mean':>6} {'var':>6}  question")
for c in consistency:
    print(f"{str(c['test_id'])[:36]:<36} {c['score_mean']!s:>6} {c['score_variance']!s:>6}  {c['question'][:70]}")


## 11. Prompt Version Control Log
Three iterations of each system prompt. The **v3** versions are the ones the agent runs above.

### Planner Prompt
**v1 — initial.** *"You are a financial planner. Decide which tools to call and answer the question."*  &nbsp;⟶ **failure mode:** the model called `query_financials` with hallucinated table names (`bank_accounts`, `payroll`) and didn't separate planning from execution. Output mixed prose with code-like `tool_call(...)` strings.

**v2 — added explicit plan format and tool list.** *"Return a numbered plan listing tool names from this set: query_financials, calculate_runway, simulate_scenario."* &nbsp;⟶ **failure mode:** still freelanced numbers in the plan ('runway will be ~5 months'), and treated every ambiguous question as something to answer rather than something to clarify. Out-of-range dates were silently ignored.

**v3 — final** (`PLANNER_SYSTEM_V3` above). Three named actions (`execute` / `clarify` / `out_of_scope`), explicit JSON output schema, explicit decision rules ordered by priority, and a hard rule to never invent numbers in the plan. **What stabilized it:** the priority-ordered decision rules — the model now reliably routes to `clarify` when a salary or delay is missing, and to `out_of_scope` when a date falls outside the dataset.

### Executor Prompt
**v1 — initial.** *"Use the tools to answer the user's question."* &nbsp;⟶ **failure mode:** stated runway figures with no tool call at all (pure hallucination). Skipped sensitivity analysis. Final answers were unstructured prose so downstream parsing was unreliable.

**v2 — added ReAct-style think-before-call instruction and an `<answer>` tag.** &nbsp;⟶ **failure mode:** the model would compute runway *itself* in the `<answer>` block instead of citing the tool's number, and tool errors caused it to give up rather than retry with corrected inputs.

**v3 — final** (`EXECUTOR_SYSTEM_V3` above). Hard *Numeric Integrity* rule ("if you state a number that isn't in a tool result, you have failed"), explicit error-recovery instruction, mandatory *Numbers / Trade-offs / Risks / Recommendation* sections, and a *Conservative Bias* clause for irreversible decisions. **What stabilized it:** anchoring numeric integrity in *failure language* ("you have failed") moved the model from soft-suggestion to compliant behavior.

### Judge Prompt
**v1 — initial.** *"Grade this agent's answer 1-5 on accuracy and reasoning."* &nbsp;⟶ **failure mode:** judge averaged ~4.5/5 on every run, including ones where the agent hallucinated numbers. No citation requirement meant the judge made up its own justifications.

**v2 — added 6-axis rubric with separate standard vs. custom dimensions.** &nbsp;⟶ **failure mode:** judge still rewarded plausible-sounding answers without checking whether the runway figure matched ground truth, and varied widely run-to-run.

**v3 — final** (`JUDGE_SYSTEM_V3` above). **Required trace citations** for every score ("Do not award full marks without a citation"), ground-truth values injected into the user message for `calculation_accuracy`, and a strict JSON-only output requirement. **What stabilized it:** the citation requirement collapses the judge's variance — it can no longer give a 5/5 without grounding the score in a specific trace step, which forces it to actually read the trace.

## 12. Architecture Decision Records (ADRs)
### ADR-1: Model Selection
**Decision:** Sonnet 4.6 for Planner & Judge; Haiku 4.5 for Executor.
**Why:** Planning and grading require strong reasoning + adherence to a structured output spec — Sonnet wins on both. Execution is mostly tool dispatch + summarization once the plan is set; Haiku is ~3× cheaper at comparable tool-use quality, and the cost shows up in 50-variant evaluation sweeps.
**Tradeoff:** Haiku occasionally formats the `<answer>` block more loosely than Sonnet would. We compensate with the strict `EXECUTOR_SYSTEM_V3` formatting rules, and the success classifier rejects any answer missing the `<answer>…</answer>` envelope.

### ADR-2: AI vs Rule-Based Components
| Component | LLM or Code? | Justification |
|---|---|---|
| Runway math | **Code** (`calculate_runway`) | Determinism, audit trail. An LLM doing arithmetic is a regression risk. |
| Scenario math | **Code** (`simulate_scenario`) | Same. The LLM names the action; code does the math. |
| Action enum | **Code** (`KNOWN_ACTIONS`) | Whitelist prevents the agent from inventing actions like `"set_company_on_fire"`. |
| Plan routing (clarify / execute / out-of-scope) | **LLM** (Planner) | Requires natural-language interpretation of vague intent. |
| Risk synthesis | **LLM** (Executor `<Risks>` block) | Selecting which uncertainty matters for *this* user is a semantic judgment. |
| Success classification | **Code** (`_classify_success`) | Pure structural check; we don't want the agent grading itself. |

### ADR-3: State Strategy
**Decision:** Stateless tasks — every user question creates a fresh `Trace` and `CostTracker`; no history is carried across tasks.
**Why:** Cash-flow advice is decision-shot, not conversational. Carrying history risks the agent reusing stale tool results when underlying data has changed. Reproducibility for evaluation is also easier with stateless tasks.
**What we lose:** follow-up questions can't reference "the runway you just calculated" — they re-run.

### ADR-4: Error Handling / Graceful Degradation
Three layers:
1. **Tool layer:** every tool returns `{status: 'error', error: ...}` instead of raising. Tools never crash the loop.
2. **Executor layer:** the system prompt explicitly tells the model to read tool errors and either retry with corrected inputs or surface the gap to the user. Empirically this kicks in most often when the agent guesses a category name (e.g. `"marketing"` vs `"Marketing"`); the error envelope lists `available` categories and the next call succeeds.
3. **Guardrail layer:** if `max_iterations`/`max_total_tokens`/`max_wall_seconds` trip, the run stops cleanly with `stopped_reason ∈ {max_iterations, token_budget, wall_clock}` and an explicit `guardrail` step in the trace. The success classifier marks these as failures so they don't contaminate the success-rate metric.

## 13. Architecture Diagram
```mermaid
flowchart TD
    U[User question] --> P[Planner Agent<br/>Sonnet 4.6]
    P -->|action=clarify| C[Clarify response]
    P -->|action=out_of_scope| O[Out-of-scope response]
    P -->|action=execute| E[Executor Agent<br/>Haiku 4.5]
    E --> L{tool_use loop<br/>guardrails:<br/>iters/tokens/wall}
    L --> T1[query_financials<br/>SQLite SELECT]
    L --> T2[calculate_runway<br/>pandas]
    L --> T3[simulate_scenario<br/>pandas + verdict]
    T1 --> L
    T2 --> L
    T3 --> L
    L -->|end_turn| A[Final answer<br/><Numbers/Trade-offs/Risks/Rec>]
    A --> J[Judge Agent<br/>Sonnet 4.6<br/>+ ground truth + citations]
    J --> R[Verdict + 6-axis scores]
    classDef ai fill:#e6f0ff,stroke:#4a80c0;
    classDef rule fill:#fff4e6,stroke:#c08040;
    class P,E,J ai;
    class T1,T2,T3,L rule;
```

## 14. Red-Team Findings & Mitigations
What we tried to break, what broke, and how we patched it.

| Attack / Edge case | Original behavior | Mitigation |
|---|---|---|
| **Out-of-range date** *("Q3 2023?")* | Planner v1 would call `query_financials` with a 2023 filter, get zero rows back, then fabricate a guess. | Planner v3 has a priority-ordered decision rule that routes to `out_of_scope` *before* generating a plan. |
| **Binary pressure** *("Just yes or no")* | Executor v2 capitulated and gave a one-word answer with no numbers. | Executor v3 has a *Conservative Bias* clause requiring a runway figure plus assumptions on every answer; the format check rejects answers without an `<answer>` block. |
| **Hallucinated numbers** | Executor v1/v2 sometimes stated runway numbers that didn't match any tool output (especially under combined scenarios). | Executor v3: "if you state a number that isn't in a tool result, you have failed." Judge v3 enforces this via `hallucination_check` with required trace citation. |
| **Wrong category casing** *("Marketing" vs "marketing")* | Tool returned a generic error and the executor gave up. | Tool now returns `{available: [...]}` so the executor can self-correct. Verified: the model retries with the correct casing on the next iteration. |
| **Missing parameter** *("Should I hire someone?" with no salary)* | Executor invented `$120k`. | Planner v3 routes to `clarify` when a material number is missing. The executor never sees the underspecified plan. |
| **Infinite loop** *(via deliberately ambiguous prompts)* | Earlier iterations hit 30+ tool calls. | `max_iterations=7` + `max_total_tokens=15_000` + `max_wall_seconds=60` — three independent kill switches. Trips are logged and counted as failures.

**Known unfixed weakness:** when the agent compares >3 scenarios in a single answer, it sometimes drops the explicit safety-threshold flag for one of them. We log this as a `scenario_completeness < 5` in the Judge output rather than hide it. Future fix: a structured-output mode forcing a row-per-scenario JSON, then format that into prose.

## 15. Burn-Rate Projection (FinOps)
From the eval pass above we can compute realistic per-run economics for the three operational states the proposal review asked for:
- **Successful run** — Planner `execute` → ~3-5 tool calls → final answer.
- **Truncated run** — guardrail trip; `stopped_reason ∈ {max_iterations, token_budget, wall_clock}`.
- **Ambiguity loop** — Planner `clarify` (zero tool calls, one Planner call only).
- **Out-of-scope** — Planner `out_of_scope` (zero tool calls).

In [ ]:
burn_buckets = {'success': [], 'truncated': [], 'clarify': [], 'out_of_scope': []}
for r in _eval_results:
    if r.stopped_reason in {'max_iterations','token_budget','wall_clock'}:
        burn_buckets['truncated'].append(r)
    elif r.stopped_reason == 'clarify':
        burn_buckets['clarify'].append(r)
    elif r.stopped_reason == 'out_of_scope':
        burn_buckets['out_of_scope'].append(r)
    elif r.success:
        burn_buckets['success'].append(r)
    else:
        burn_buckets['truncated'].append(r)  # other failure modes counted as truncated

print(f"{'bucket':<14} {'n':>4} {'avg_cost':>10} {'avg_tokens':>11} {'avg_wall_s':>11}")
for name, runs in burn_buckets.items():
    if not runs:
        print(f'{name:<14} {0:>4}    -          -          -')
        continue
    n = len(runs)
    avg_cost = sum(r.cost.total_cost_usd for r in runs) / n
    avg_tok = sum(r.cost.total_input_tokens + r.cost.total_output_tokens for r in runs) / n
    avg_wall = sum(r.trace.to_dict()['wall_seconds'] for r in runs) / n
    print(f'{name:<14} {n:>4} ${avg_cost:>9.4f} {avg_tok:>11.0f} {avg_wall:>11.2f}')

# Ambiguity-loop overhead: a user whose question triggers 'clarify' and then
# resubmits incurs one clarify run + one success run before getting an answer.
clarify_runs = burn_buckets['clarify']
success_runs = burn_buckets['success']
if clarify_runs and success_runs:
    avg_clarify = sum(r.cost.total_cost_usd for r in clarify_runs) / len(clarify_runs)
    avg_success = sum(r.cost.total_cost_usd for r in success_runs) / len(success_runs)
    print(f'\nAmbiguity-loop total (clarify + re-run): '
          f'${avg_clarify + avg_success:.4f} avg per resolved question')
    print(f'  Clarify overhead alone: ${avg_clarify:.4f}  '
          f'(approx {avg_clarify/(avg_clarify+avg_success)*100:.0f}% of full run cost)')


## 16. Save Artifacts
Persists all traces, judge verdicts, and the aggregated report so they can be checked in alongside the notebook for the final submission.

In [ ]:
import json, os, datetime as dt

out_dir = 'artifacts'
os.makedirs(out_dir, exist_ok=True)
stamp = dt.datetime.now().strftime('%Y%m%d_%H%M%S')

with open(f'{out_dir}/seed_runs_{stamp}.json', 'w') as f:
    json.dump([r.to_dict() for r in _seed_results], f, indent=2, default=str)
with open(f'{out_dir}/seed_verdicts_{stamp}.json', 'w') as f:
    json.dump(_seed_verdicts, f, indent=2, default=str)
with open(f'{out_dir}/eval_runs_{stamp}.json', 'w') as f:
    json.dump([r.to_dict() for r in _eval_results], f, indent=2, default=str)
with open(f'{out_dir}/eval_verdicts_{stamp}.json', 'w') as f:
    json.dump(_eval_verdicts, f, indent=2, default=str)
with open(f'{out_dir}/consistency_{stamp}.json', 'w') as f:
    json.dump(consistency, f, indent=2, default=str)
with open(f'{out_dir}/summary_{stamp}.json', 'w') as f:
    json.dump(summary, f, indent=2, default=str)
with open(f'{out_dir}/ground_truth_{stamp}.json', 'w') as f:
    json.dump(GROUND_TRUTH, f, indent=2, default=str)

print('Wrote artifacts to', out_dir)
for fn in sorted(os.listdir(out_dir)):
    sz = os.path.getsize(os.path.join(out_dir, fn))
    print(f'  {fn:<40} {sz:>9,d} bytes')
